In [1]:
import os
import pandas as pd

In [2]:
raw_data_dir = "/Users/joshua/Developer/civetqc/data/raw"

In [3]:
import sys
sys.path.append("/Users/joshua/Developer/civetqc")
sys.path.append("/Users/joshua/Developer/CognitiveSubtypes/")

In [4]:
from civetqc.data.study import Study

In [5]:
study = Study(name="NUSDAST")

In [6]:
df = study.get_text_features()

In [7]:
class Study:
    
    def __init__(self, name):
        
        self.name = name
        self.path_civet_data = os.path.join(raw_data_dir, self.name, f"{self.name}_civet_data.csv")
        self.path_qc_data = os.path.join(raw_data_dir, self.name, f"{self.name}_QC.csv")
        
        for filepath in [self.path_civet_data, self.path_qc_data]:
            if not os.path.exists(filepath):
                raise FileNotFoundError("Missing expected file: " + filepath)
        
        self.civet_data = self.load_civet_data()
        self.qc_data = self.load_qc_data()
        self.data = pd.merge(self.civet_data, self.qc_data).dropna()
        assert len(self.data["ID"]) == len(self.data["ID"].unique())
        self.n_scans = len(self.data["ID"])
        
    def load_civet_data(self):
        df = pd.read_csv(self.path_civet_data)
        df_cols = ['ID', 'XSTEP', 'YSTEP', 'ZSTEP', 'MASK_ERROR', 'WM_PERCENT', 'GM_PERCENT', 
                   'CSF_PERCENT', 'SC_PERCENT', 'BRAIN_VOL', 'CEREBRUM_VOL', 'CORTICAL_GM', 
                   'WHITE_VOL', 'SUBGM_VOL', 'SC_VOL', 'CSF_VENT_VOL', 'LEFT_WM_AREA',
                   'LEFT_MID_AREA', 'LEFT_GM_AREA', 'RIGHT_WM_AREA', 'RIGHT_MID_AREA', 
                   'RIGHT_GM_AREA', 'GI_LEFT', 'GI_RIGHT', 'LEFT_INTER', 'RIGHT_INTER', 
                   'LEFT_SURF_SURF', 'RIGHT_SURF_SURF', 'LAPLACIAN_MIN', 'LAPLACIAN_MAX', 
                   'LAPLACIAN_MEAN', 'GRAY_LEFT_RES', 'GRAY_RIGHT_RES']
        assert [col for col in df_cols if col not in df.columns] == []
        return df[df_cols]
    
    def load_qc_data(self):
        return pd.read_csv(self.path_qc_data)

In [8]:
studies = {name: Study(name) for name in ["FEP", "INSIGHT", "LAM", "TOPSY", "NUSDAST"]}

In [9]:
[study.n_scans for study in studies.values()]

[456, 182, 333, 115, 130]

In [10]:
sum([study.n_scans for study in studies.values()])

1216